# HDD 24 — Frozen streaming inference replay and telemetry contract

Notebook 23 completed controlled sensitivity on 1,000 stop-drive windows. The model currently exposes three frozen detector channels; regime and drift outputs have **not** passed an operational gate. This notebook defines a minimal intake contract for the team's SMART collection agent and checks whether 30 daily records produce exactly the same scores as Notebook 18’s offline Q3 batch path.

The live agent and frontend repositories are not present here, so this is a research-compatible contract and executable replay, not a claim that their code has already been integrated. This notebook reads only 30 Q3 days and precomputed **unlabeled** evaluation score rows for a 50-drive parity check. No failure flags, Q4 or sealed June outcomes are opened. Threshold flags are research candidates, not a failure probability or automatic self-healing command.

In [1]:
from pathlib import Path
from collections import deque
from datetime import datetime,timezone
import hashlib,json,re,zipfile
import numpy as np
import pandas as pd
import torch
from torch import nn
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
TRAIN_ID='q3_five_model_masked_transformer_v1'
SCORE_ID='q3_five_model_transformer_scoring_v1'
SENS_ID='q3_frozen_detector_synthetic_sensitivity_v1'
REPLAY_ID='q3_frozen_streaming_inference_replay_v1'
TRAIN=COMPONENT_ROOT/'results'/RUN_ID/TRAIN_ID
TRAINED=COMPONENT_ROOT/'models/trained'/RUN_ID/TRAIN_ID
SCORE=COMPONENT_ROOT/'results'/RUN_ID/SCORE_ID
SENS=COMPONENT_ROOT/'results'/RUN_ID/SENS_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/REPLAY_ID
OUT.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
trained=read_json(TRAIN/'encoder_training_complete.json')
scored=read_json(SCORE/'transformer_scoring_complete.json')
sensitive=read_json(SENS/'synthetic_sensitivity_complete.json')
assert all(x['analysis_complete'] for x in [trained,scored,sensitive])
assert not trained['failure_labels_used'] and not scored['failure_labels_used'] and not sensitive['failure_labels_used']
assert not trained['future_test_outcomes_opened'] and not scored['future_test_opened'] and not sensitive['future_test_opened']
assert not trained['sealed_June_test_opened'] and not scored['sealed_June_test_opened'] and not sensitive['sealed_June_test_opened']
train_settings=read_json(TRAIN/'settings.json')
score_settings=read_json(SCORE/'settings.json')
prep_path=TRAINED/'preprocessing.json';prep=read_json(prep_path)
checkpoint_path=TRAINED/'best_model.pt'
assert sha256(prep_path)==trained['preprocessing_sha256']==score_settings['preprocessing_sha256']
assert sha256(checkpoint_path)==trained['best_checkpoint_sha256']==score_settings['checkpoint_sha256']
assert sha256(SCORE/'calibrated_thresholds.json')==scored['thresholds_sha256']
assert sha256(Q3_ZIP)==score_settings['q3_zip_sha256']
thresholds=read_json(SCORE/'calibrated_thresholds.json')
models=prep['models'];assert len(models)==5 and set(models)==set(thresholds)
FEATURES=prep['features']
assert FEATURES==['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
assert score_settings['hidden_final_day_features']==['smart_5_raw','smart_194_raw','smart_197_raw','smart_198_raw']
assert score_settings['detectors']==['temp_error','sector_error','joint_error']
HIDDEN=[0,2,3,4];model_index={m:i for i,m in enumerate(models)}
END_DATE='2021-09-15';DATES=pd.date_range(end=END_DATE,periods=30).strftime('%Y-%m-%d').tolist()
CONTRACT={'schema_version':'research_smart_daily_v1',
          'input':{'serial_number':'nonempty string','model':'one of the five frozen HDD models',
                   'date':'ISO 8601 calendar day YYYY-MM-DD; one record per drive/day',
                   **{key:'finite nonnegative observed number' for key in FEATURES}},
          'readiness':'30 consecutive valid daily observations per drive; gap resets window',
          'duplicate':'identical same-day record is idempotent; conflicting record rejected',
          'ordering':'per-drive dates strictly increasing; late/backfilled days rejected',
          'output':['serial_number','model','end_date','status','history_days','scores',
                    'thresholds','flags','sensor_residuals','smart_evidence','model_version'],
          'risk_semantics':'candidate anomaly flags; not failure probability or automatic healing',
          'regime_status':'unavailable','drift_status':'unavailable',
          'required_for_real_agent_integration':['confirm agent field names and units',
                'daily timestamp timezone and end-of-day policy','device identity and model mapping',
                'missing-value semantics','deduplication and storage policy']}
SETTINGS={'checkpoint_sha256':sha256(checkpoint_path),'preprocessing_sha256':sha256(prep_path),
          'thresholds_sha256':sha256(SCORE/'calibrated_thresholds.json'),
          'sensitivity_complete_sha256':sha256(SENS/'synthetic_sensitivity_complete.json'),
          'q3_zip_sha256':sha256(Q3_ZIP),'replay_end_date':END_DATE,'replay_drives_per_model':10,
          'contract':CONTRACT,'failure_labels_used':False,'future_test_opened':False}
settings_path=OUT/'settings.json'
if settings_path.exists():assert read_json(settings_path)==SETTINGS,'Change REPLAY_ID before changing the contract.'
else:settings_path.write_text(json.dumps(SETTINGS,indent=2))
(OUT/'telemetry_contract.json').write_text(json.dumps(CONTRACT,indent=2))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Frozen artifacts verified | device:',DEVICE,'| replay window:',DATES[0],END_DATE)

Frozen artifacts verified | device: cpu | replay window: 2021-08-17 2021-09-15


## 1. Restore Notebook 18’s exact frozen inference path

The class and preprocessing match Notebook 18, including the final-day mask and masked adjacent-day differences. No training occurs. A daily row is scored only when all 30 observations are consecutive and valid. Scores/flags remain separate for temperature, sector, and joint reconstruction; a regime label and drift event are explicitly unavailable.

In [2]:
class MaskedTelemetryTransformer(nn.Module):
    def __init__(self,n_models,width=64,heads=4,layers=2,feedforward=128,dropout=.1):
        super().__init__()
        self.input=nn.Linear(15,width)
        self.model_embedding=nn.Embedding(n_models,width)
        self.position=nn.Parameter(torch.zeros(1,30,width))
        nn.init.normal_(self.position,std=.02)
        block=nn.TransformerEncoderLayer(d_model=width,nhead=heads,dim_feedforward=feedforward,
                                          dropout=dropout,batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(block,num_layers=layers,enable_nested_tensor=False)
        self.output=nn.Linear(width,5)
    def forward(self,x,model_id,hide):
        observed=(~hide).to(x.dtype)
        masked=x.masked_fill(hide,0)
        delta=torch.zeros_like(x)
        delta[:,1:,:]=x[:,1:,:]-x[:,:-1,:]
        delta_visible=observed.clone()
        delta_visible[:,1:,:]=observed[:,1:,:]*observed[:,:-1,:]
        delta=delta*delta_visible
        h=self.input(torch.cat([masked,observed,delta],dim=-1))
        h=h+self.model_embedding(model_id)[:,None,:]+self.position
        return self.output(self.encoder(h))
network=MaskedTelemetryTransformer(len(models),train_settings['width'],train_settings['heads'],
                                   train_settings['layers'],train_settings['feedforward'],train_settings['dropout']).to(DEVICE)
checkpoint=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
assert checkpoint['models']==models and checkpoint['preprocessing_sha256']==sha256(prep_path)
network.load_state_dict(checkpoint['state_dict']);network.eval()
@torch.no_grad()
def frozen_errors(window,model):
    assert window.shape==(30,5) and model in models
    logged=np.log1p(window.astype(np.float64))
    center=np.asarray(prep['center'][model],dtype=np.float64)
    scale=np.asarray(prep['scale'][model],dtype=np.float64)
    scaled=((logged-center)/scale).astype(np.float32)
    assert np.isfinite(scaled).all()
    x=torch.from_numpy(scaled[None]).to(DEVICE)
    m=torch.tensor([model_index[model]],device=DEVICE)
    hide=torch.zeros_like(x,dtype=torch.bool);hide[:,-1,HIDDEN]=True
    residual=(network(x,m,hide)[:,-1,HIDDEN]-x[:,-1,HIDDEN]).square()
    out=residual.cpu().numpy()[0].astype(np.float32)
    assert np.isfinite(out).all()
    return out
class FrozenDailyScorer:
    def __init__(self):
        self.history={};self.last={};self.model_for={};self.last_values={}
    def ingest(self,row):
        serial=row['serial_number'];model=row['model'];date=row['date']
        if not isinstance(serial,str) or not serial:raise ValueError('serial_number must be a nonempty string')
        if model not in models:raise ValueError('unknown model')
        if not isinstance(date,str) or not re.fullmatch(r'\d{4}-\d{2}-\d{2}',date):raise ValueError('date must be YYYY-MM-DD')
        day=pd.Timestamp(date)
        if str(day.date())!=date:raise ValueError('invalid date')
        values=np.asarray([row[f] for f in FEATURES],dtype=np.float64)
        if not(np.isfinite(values).all() and (values>=0).all()):raise ValueError('all five observed SMART values must be finite and nonnegative')
        values=values.astype(np.float32)
        if serial in self.model_for and self.model_for[serial]!=model:raise ValueError('model changed for serial')
        prior=self.last.get(serial)
        if prior is not None and day==prior:
            if np.array_equal(values,self.last_values[serial]):return {'status':'duplicate','serial_number':serial,'model':model,'end_date':date}
            raise ValueError('conflicting duplicate day')
        if prior is not None and day<prior:raise ValueError('out-of-order record')
        gap=bool(prior is not None and (day-prior).days!=1)
        if gap or serial not in self.history:self.history[serial]=deque(maxlen=30)
        previous=self.last_values.get(serial) if not gap else None
        self.history[serial].append(values)
        self.last[serial]=day;self.last_values[serial]=values;self.model_for[serial]=model
        h=self.history[serial]
        if len(h)<30:
            return {'status':'warming','serial_number':serial,'model':model,'end_date':date,
                    'history_days':len(h),'reset_reason':'gap' if gap else None}
        window=np.stack(h)
        errors=frozen_errors(window,model)
        scores={'temp_error':float(errors[1]),
                'sector_error':float(errors[[0,2,3]].mean()),'joint_error':float(errors.mean())}
        flags={name:bool(score>thresholds[model][name]) for name,score in scores.items()}
        sector_positive=bool(np.any(values[[0,3,4]]>0))
        sector_delta=float(np.maximum(values[[0,3,4]]-previous[[0,3,4]],0).sum()) if previous is not None else None
        return {'status':'scored','serial_number':serial,'model':model,'end_date':date,
                'history_days':30,'scores':scores,'thresholds':thresholds[model],
                'flags':flags,'sensor_residuals':dict(zip(['smart_5_raw','smart_194_raw','smart_197_raw','smart_198_raw'],map(float,errors))),
                'smart_evidence':{'sector_positive':sector_positive,'sector_delta':sector_delta},
                'regime_id':None,'drift_event':None,
                'model_version':sha256(checkpoint_path)}
print('Frozen daily scorer ready; anomaly channels remain separate.')

Frozen daily scorer ready; anomaly channels remain separate.


## 2. Replay 50 evaluation drives and verify offline parity

Select ten reference score rows per model for September 15. Read only serial, model, date and five SMART features from the 30 relevant Q3 files. Stream each day in order. The result must match Notebook 18’s sensor residuals, three scores, and three flags. This tests the proposed agent-shaped intake on the exact research data; no failure outcomes are consulted.

In [3]:
reference=pd.read_parquet(SCORE/'daily_scores'/f'{END_DATE}.parquet')
reference=reference.loc[reference.role.eq('evaluation')].copy()
expected=[]
for model in models:
    g=reference.loc[reference.model.eq(model)].sort_values('serial_number')
    assert len(g)>=SETTINGS['replay_drives_per_model']
    expected.append(g.head(SETTINGS['replay_drives_per_model']))
expected=pd.concat(expected,ignore_index=True)
assert len(expected)==len(models)*SETTINGS['replay_drives_per_model']
wanted=set(expected.serial_number)
with zipfile.ZipFile(Q3_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in DATES:
            assert match.group(1) not in members
            members[match.group(1)]=info.filename
assert set(members)==set(DATES)
stream=FrozenDailyScorer();output={}
with zipfile.ZipFile(Q3_ZIP) as z:
    for date in DATES:
        records=[];seen=set()
        with z.open(members[date]) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number','model']+FEATURES,
                                     dtype={'serial_number':str},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(wanted)]
                assert not part.serial_number.duplicated().any() and seen.isdisjoint(part.serial_number)
                seen.update(part.serial_number)
                records.extend(part.to_dict('records'))
        assert seen==wanted,(date,len(seen),len(wanted))
        for row in sorted(records,key=lambda x:x['serial_number']):
            assert str(row['date'])==date
            result=stream.ingest(row)
            if date!=END_DATE:assert result['status']=='warming'
            else:
                assert result['status']=='scored'
                output[row['serial_number']]=result
assert set(output)==wanted
matches=[]
for row in expected.itertuples(index=False):
    actual=output[row.serial_number]
    assert actual['model']==row.model and actual['end_date']==END_DATE
    for name,col in [('smart_5_raw','error_5'),('smart_194_raw','error_194'),
                     ('smart_197_raw','error_197'),('smart_198_raw','error_198')]:
        np.testing.assert_allclose(actual['sensor_residuals'][name],getattr(row,col),rtol=1e-5,atol=1e-6)
    for name in ['temp_error','sector_error','joint_error']:
        np.testing.assert_allclose(actual['scores'][name],getattr(row,name),rtol=1e-5,atol=1e-6)
        assert actual['flags'][name]==bool(actual['scores'][name]>thresholds[row.model][name])
    matches.append({'serial_number':row.serial_number,'model':row.model,'end_date':END_DATE,
                    'temp_error':actual['scores']['temp_error'],'sector_error':actual['scores']['sector_error'],
                    'joint_error':actual['scores']['joint_error'],
                    'temp_error_flag':actual['flags']['temp_error'],
                    'sector_error_flag':actual['flags']['sector_error'],
                    'joint_error_flag':actual['flags']['joint_error']})
parity=pd.DataFrame(matches)
parity.to_csv(OUT/'streaming_parity_50_drives.csv',index=False)
print('Exact reference parity within float tolerance:',len(parity),'drives')
display(parity.groupby('model').agg(drives=('serial_number','size'),sector_flags=('sector_error_flag','sum'),temp_flags=('temp_error_flag','sum')))

Exact reference parity within float tolerance: 50 drives


,drives,sector_flags,temp_flags
model,,,
HGST HMS5C4040BLE640,10,0,0
ST12000NM0008,10,0,0
ST4000DM000,10,0,0
ST8000NM0055,10,0,0
TOSHIBA MG07ACA14TA,10,0,0


## 3. Verify intake edge cases and save example payloads

The local scorer explicitly handles warmup, gaps, exact duplicate delivery, conflicting duplicate delivery, invalid values, model changes and late records. This notebook does not impose persistence or a single combined alert; those require an agreed operational burden and separate test. The JSON examples help the frontend and agent teams align field names without guessing from model internals.

In [4]:
example_model=expected.iloc[0].model
example_serial='synthetic-contract-check'
test=FrozenDailyScorer()
value={f:float(v) for f,v in zip(FEATURES,[0,1000,28,0,0])}
def payload(date,serial=example_serial,model=example_model,**overrides):
    return {'serial_number':serial,'model':model,'date':date,**{**value,**overrides}}
first=test.ingest(payload('2021-09-01'))
assert first['status']=='warming' and first['history_days']==1
assert test.ingest(payload('2021-09-01'))['status']=='duplicate'
try:test.ingest(payload('2021-09-01',smart_194_raw=29))
except ValueError as e:assert 'conflicting duplicate' in str(e)
else:raise AssertionError('conflicting duplicate accepted')
second=test.ingest(payload('2021-09-03'))
assert second['status']=='warming' and second['history_days']==1 and second['reset_reason']=='gap'
for bad in [payload('2021-09-02'),payload('2021-09-04',smart_194_raw=-1),
            payload('2021-09-04',model='unknown')]:
    try:test.ingest(bad)
    except ValueError:pass
    else:raise AssertionError('Invalid intake accepted')
example_input=payload(END_DATE,serial=expected.iloc[0].serial_number,model=example_model)
example_output=output[expected.iloc[0].serial_number]
(OUT/'agent_example_input.json').write_text(json.dumps(example_input,indent=2))
(OUT/'scorer_example_output.json').write_text(json.dumps(example_output,indent=2))
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'settings_sha256':sha256(settings_path),'contract_sha256':sha256(OUT/'telemetry_contract.json'),
        'parity_sha256':sha256(OUT/'streaming_parity_50_drives.csv'),
        'parity_drives':len(parity),'edge_case_checks_passed':True,
        'failure_labels_used':False,'future_test_opened':False,'sealed_June_test_opened':False,
        'regime_and_drift_enabled':False,'automatic_healing_enabled':False,
        'limitations':['agent field schema and timestamp convention must be confirmed against actual agent',
                       'only fifty drives and one replay endpoint tested',
                       'threshold flags are research candidates; operational alert policy not approved',
                       'no persistence/deduplication of repeated anomaly episodes']}
(OUT/'streaming_replay_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 24 COMPLETE — frozen stream/batch parity and intake contract verified.')
print('Q4 and sealed June unopened. Artifacts:',OUT)

NOTEBOOK 24 COMPLETE — frozen stream/batch parity and intake contract verified.
Q4 and sealed June unopened. Artifacts: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_frozen_streaming_inference_replay_v1
